# 💳 Ejercicios: Visa
# Programas de cashback por tipo de tarjeta

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales de Visa.

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué nivel de tarjeta
### deja más cashback
### neto al cliente?

Cashback menos cuota anual. A veces
la cuota se come la recompensa.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/visa_recompensas.csv'
)

df['neto'] = (
  df['cashback'] - df['cuota_anual']
)

r = (
  df
  .groupby('nivel')
  .agg(
    cashback=('cashback', 'mean'),
    cuota=('cuota_anual', 'mean'),
    neto=('neto', 'mean'),
  )
  .round(1)
  .sort_values('neto',
    ascending=False)
)

print('Cashback neto por nivel:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué % del gasto
### vuelve como cashback
### en cada nivel?

La tasa efectiva = cashback / gasto.
Revela el valor real del programa.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/visa_recompensas.csv'
)

g = (
  df
  .groupby('nivel')
  .agg(
    gasto=('gasto_anual', 'sum'),
    cb=('cashback', 'sum'),
  )
)

g['tasa_%'] = (
  g['cb'] / g['gasto'] * 100
).round(2)

print('Tasa efectiva de cashback:')
print(
  g['tasa_%']
  .sort_values(ascending=False)
)

### 💼 PREGUNTA ENTREVISTA:
### ¿A partir de qué
### gasto compensa pagar
### la cuota anual?

Umbral de rentabilidad: gasto donde
cashback iguala la cuota, por nivel.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/visa_recompensas.csv'
)

df['tasa'] = (
  df['cashback'] / df['gasto_anual']
)

g = (
  df
  .groupby('nivel')
  .agg(
    tasa=('tasa', 'mean'),
    cuota=('cuota_anual', 'mean'),
  )
)

g['umbral'] = (
  g['cuota'] / g['tasa']
).round(0)

print('Gasto mínimo rentable (€):')
print(
  g['umbral']
  .sort_values()
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué perfil de uso
### gasta más por
### nivel? (cruce)

Pivot uso x nivel sobre gasto medio
para enfocar campañas de marketing.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/visa_recompensas.csv'
)

r = pd.pivot_table(
  df,
  index='uso',
  columns='nivel',
  values='gasto_anual',
  aggfunc='mean',
)

r = (r / 1000).round(1)

print('Gasto medio (k€) uso x nivel:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuánto gastan los
### top 20% de tarjetas
### vs el resto? (Pareto)

Regla 80/20: ver qué parte del gasto
concentra el 20% de clientes top.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/visa_recompensas.csv'
)

df['grupo'] = pd.qcut(
  df['gasto_anual'],
  q=[0, 0.8, 1.0],
  labels=['Resto 80%', 'Top 20%'],
)

r = (
  df
  .groupby('grupo', observed=True)
  ['gasto_anual']
  .sum()
)

pct = (r / r.sum() * 100).round(1)

print('Gasto total por grupo:')
print((r / 1e6).round(2))
print('\n% del gasto:')
print(pct)